# IOAI — 2025 Summer Online University Admission (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2025/nyari-online/adatok >/dev/null && git checkout -q
import shutil, glob
for f in glob.glob('haio/2025/nyari-online/adatok/*'): shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# University Admission Prediction (Egyetemi Felvételi)

> **HAIO 2025 — Summer Online Qualifier (ML)**

Predict the probability of university admission (`Felvételi Eredmény`). Score = **ROC-AUC**.
The official `test.csv` is unlabelled (Kaggle), so we grade on a **held-out split** (`index % 5`) of `train.csv`; **Submit** `submission.csv` (`ID,Felvételi Eredmény` probability).

Baseline: HistGradientBoosting with ordinal-encoded categoricals.

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
def prep(Xtr, Xte):
    cat = [c for c in Xtr.columns if Xtr[c].dtype == 'object']
    num = [c for c in Xtr.columns if c not in cat]
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    Atr = Xtr[cat].astype('object').where(Xtr[cat].notna(), 'NA')
    Ate = Xte[cat].astype('object').where(Xte[cat].notna(), 'NA')
    Ctr = enc.fit_transform(Atr); Cte = enc.transform(Ate)
    import numpy as np
    return (np.hstack([Xtr[num].to_numpy(float), Ctr]),
            np.hstack([Xte[num].to_numpy(float), Cte]))

df = pd.read_csv('train.csv').reset_index(drop=True)
TARGET = 'Felvételi Eredmény'
is_val = (df.index % 5 == 0)
tr, va = df[~is_val], df[is_val]
Xtr, Xva = prep(tr.drop(columns=['ID', TARGET]), va.drop(columns=['ID', TARGET]))
ytr, yva = tr[TARGET].to_numpy(), va[TARGET].to_numpy()
len(tr), len(va)

## Train on the train split, predict the held-out val → `submission.csv`

In [ ]:
clf = HistGradientBoostingClassifier(random_state=42).fit(Xtr, ytr)
proba = clf.predict_proba(Xva)[:, 1]
print(f'Held-out ROC-AUC: {roc_auc_score(yva, proba):.4f}')
pd.DataFrame({'ID': va['ID'].to_numpy(), TARGET: proba}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)